<a href="https://colab.research.google.com/github/JudeTulel/0g-compute-ts-starter-kit/blob/main/nllb_pokot_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rescuing a Language from the Digital Void: Fine-Tuning NLLB-200 for Pokot

In the landscape of modern AI, 'low-resource' languages often get left behind. Pokot (*Pökot*), a Nilotic language spoken by over 700,000 people, is one such language. Today, we are changing that.

This notebook serves as a technical deep-dive into fine-tuning Meta's **NLLB-200 (No Language Left Behind)** model specifically for English-to-Pokot translation. We aren't just training a model; we're bridging a digital divide using a 30k-verse parallel corpus, dictionary augmentations, and modern transfer learning techniques.

### The Workflow
* **Data Harvesting:** Loading the PIPÏLIA parallel corpus and dictionary pairs.
* **Vocabulary Expansion:** Injecting Pokot-specific subwords into the NLLB tokenizer.
* **Architecture Surgery:** Resizing embeddings and freezing components for efficiency.
* **The Training Loop:** Leveraging `Seq2SeqTrainer` with label smoothing and gradient checkpointing.
* **Persistence:** Ensuring our progress is safely stored in Google Drive.

> **Pro Tip:** Before diving in, ensure your runtime is set to a **T4 GPU** to handle the 600M parameters efficiently.

In [1]:
# 1) Install pinned dependencies (no restart needed)
%pip -q install "transformers==4.46.3" "datasets>=2.20" sentencepiece "sacrebleu>=2.3" "accelerate>=0.34" pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 13.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Setting the Stage: Environment & Data Persistence

Every great machine learning project starts with a robust environment. Because Colab runtimes are ephemeral, we begin by mounting Google Drive. This ensures that our final model and checkpoints survive if the session disconnects. We'll also extract our raw data—a ZIP file containing our parallel corpus, dictionary, and monolingual text—into the local runtime for high-speed access.

In [3]:
import os, sys, random, shutil, zipfile, time
import numpy as np, pandas as pd, torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Define potential paths for the data zip
COLAB_LOCAL_ZIP = "/content/pokot_colab_data.zip"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/pokot_colab_data.zip"
DATA_ZIP = None # Initialize DATA_ZIP

LOCAL_DATA_DIR = "./data"          # used when NOT on Colab

if IN_COLAB:
    if os.path.exists(COLAB_LOCAL_ZIP):
        DATA_ZIP = COLAB_LOCAL_ZIP
        print(f"Using local Colab data zip: {DATA_ZIP}")
    elif os.path.exists(DRIVE_ZIP_PATH):
        DATA_ZIP = DRIVE_ZIP_PATH
        print(f"Using Google Drive data zip: {DATA_ZIP}")

    assert DATA_ZIP is not None, f"Upload pokot_colab_data.zip to /content/ or to Drive root; not found at {COLAB_LOCAL_ZIP} nor {DRIVE_ZIP_PATH}"

    DATA_DIR = "/content/pokot_data"
    os.makedirs(DATA_DIR, exist_ok=True)
    if not os.path.exists(f"{DATA_DIR}/parallel_corpus.csv"):
        with zipfile.ZipFile(DATA_ZIP) as z:
            z.extractall(DATA_DIR)
else:
    DATA_DIR = LOCAL_DATA_DIR
    os.makedirs(DATA_DIR, exist_ok=True) # Ensure local data dir exists

# Updated to check only the 3 available files
for f in ["parallel_corpus.csv", "pokot_mono.txt", "pokot_dict.csv"]:
    p = os.path.join(DATA_DIR, f)
    assert os.path.exists(p), f"missing {p}"
    print(f"ok {p}  ({os.path.getsize(p)/1e6:.2f} MB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using local Colab data zip: /content/pokot_colab_data.zip
ok /content/pokot_data/parallel_corpus.csv  (7.39 MB)
ok /content/pokot_data/pokot_mono.txt  (3.34 MB)
ok /content/pokot_data/pokot_dict.csv  (0.74 MB)


In [4]:
import torch
import random
import numpy as np

# 3) Config
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "facebook/nllb-200-distilled-600M"
NEW_LANG   = "pko_Latn"     # pseudo language code we register for Pokot
SRC_LANG   = "eng_Latn"     # NLLB's English code

# --- COLAB OPTIMIZATIONS ---
SMOKE_TEST = False
EPOCHS     = 8
LR         = 2e-5
BS         = 4
ACCUM      = 8
MAX_LEN    = 192
WARMUP     = 300
LABEL_SMOOTH = 0.1

USE_DICT_AUG    = True
USE_GRAMMAR_AUG = False # Set to False as file is not in zip

# Ensure output directory exists in Google Drive for persistence
OUT_DIR = "/content/drive/MyDrive/pokot_nllb"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Device: {device}")

Environment: Colab
Device: cuda


## The Data Strategy: Quality Over Quantity

When working with only 30,000 pairs, every verse counts. However, random splits are dangerous in religious or literary texts because adjacent verses often share identical phrasing. To prevent 'data leakage' and ensure our model actually learns to translate rather than memorize, we split our data by **entire books**.

We’ve held out specific sections—like the book of John and portions of the Psalms—to serve as the ultimate litmus test for our model's performance.

In [5]:
df = pd.read_csv(os.path.join(DATA_DIR, "parallel_corpus.csv"))
print("raw pairs:", len(df))

def clean(df):
    df = df.dropna(subset=["english", "pokot"]).copy()
    df["english"] = df.english.str.replace(r"\s+", " ", regex=True).str.strip()
    df["pokot"]   = df.pokot.str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[(df.english.str.len() >= 10) & (df.pokot.str.len() >= 8)]
    df = df.drop_duplicates(subset=["english", "pokot"])
    # some verses have the verse number glued to the start of text
    df["english"] = df.english.str.replace(r"^\d+[-\d\s]*", "", regex=True).str.strip()
    df["pokot"]   = df.pokot.str.replace(r"^\d+[-\d\s]*", "", regex=True).str.strip()
    return df[df.english.str.len() > 0].reset_index(drop=True)

df = clean(df)
print("clean pairs:", len(df))

mask = df.apply(lambda r: (r.book == "JHN") or (r.book == "1CO")
                or (r.book == "PSA" and 90 <= r.chapter <= 119), axis=1)
eval_df  = df[mask].reset_index(drop=True)
train_df = df[~mask].reset_index(drop=True)
print(f"train: {len(train_df):,} | eval(held-out books): {len(eval_df):,}")
df.sample(3, random_state=SEED)[["ref", "english", "pokot"]]

raw pairs: 30057
clean pairs: 29821
train: 27,848 | eval(held-out books): 1,973


,ref,english,pokot
5477,2SA.17.18,"But one day a boy happened to see them, and he...",Kïsïwa karöchïnin chane asistanka akïwö kïmwoc...
579,1CH.20.1,"The following spring, at the time of the year ...",Kintöghoghchï Yoap pipö lïk akupokyï koro pipö...
279,1CH.7.28,The territory which they took and settled incl...,Kor nyo kicheng chane akïmang kï Petel nko kan...


## Tokenization: Teaching the Model to 'Read' Pokot

Standard NLLB-200 has never seen a Pokot vowel. Without modification, the model would see Pokot text as a string of 'Unknown' tokens (`[UNK]`).

To fix this, we train a custom **SentencePiece** model on Pokot monolingual text. We then surgically inject these new subwords into the NLLB tokenizer. By adding `pko_Latn` as a registered language code, we give Pokot a first-class seat at the table.

In [6]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
UNK = tok.unk_token_id

def unk_ratio(texts, n=2000):
    tot = unks = 0
    for t in texts[:n]:
        ids = tok(t, add_special_tokens=False).input_ids
        tot += len(ids); unks += ids.count(UNK)
    return unks / max(tot, 1)

mono = [l for l in open(os.path.join(DATA_DIR, "pokot_mono.txt"), encoding="utf-8").read().split("\n") if l.strip()]
print(f"mono lines: {len(mono):,}")
print(f"unk ratio BEFORE extension: {unk_ratio(mono):.4f}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

mono lines: 30,057
unk ratio BEFORE extension: 0.0064


In [7]:
import sentencepiece as spm

SP_DIR = "/content/sp" if IN_COLAB else "./sp"
os.makedirs(SP_DIR, exist_ok=True)
SP_PREFIX = os.path.join(SP_DIR, "pokot")
mono_path = os.path.join(SP_DIR, "mono.txt")
open(mono_path, "w", encoding="utf-8").write("\n".join(mono))

spm.SentencePieceTrainer.train(
    input=mono_path, model_prefix=SP_PREFIX, vocab_size=2500,
    model_type="bpe", character_coverage=1.0,
)
sp = spm.SentencePieceProcessor(model_file=SP_PREFIX + ".model")
pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]

existing = tok.get_vocab()
skip = {"<s>", "</s>", "<unk>", "<pad>"}
new_tokens = [p for p in pieces if p not in existing and p not in skip]
added = tok.add_tokens(new_tokens + [NEW_LANG])
print(f"added {added} tokens | vocab now {len(tok):,}")

SRC_ID = tok.convert_tokens_to_ids(SRC_LANG)
TGT_ID = tok.convert_tokens_to_ids(NEW_LANG)
EOS_ID = tok.eos_token_id
PAD_ID = tok.pad_token_id
assert SRC_ID != UNK and TGT_ID != UNK, "language tokens not found after extension"
print(f"unk ratio AFTER extension: {unk_ratio(mono):.4f}")

added 1296 tokens | vocab now 257,500
unk ratio AFTER extension: 0.0000


## Model Surgery: Resizing the Embedding Layer

Since we've added over 1,000 new tokens to the tokenizer, the original model's embedding matrix is now too small. We must resize the model's 'vocabulary' layers. The existing weights for English and other languages remain intact, while the new Pokot-specific rows are initialized and ready to be learned during the training phase.

In [8]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tok))          # new rows = new Pokot pieces + pko_Latn

# generation defaults (used by Trainer's predict_with_generate AND manual calls)
model.generation_config.forced_bos_token_id = TGT_ID
model.generation_config.max_length = 256
model.generation_config.num_beams = 1
print(f"params: {model.num_parameters()/1e6:.0f}M")

# OPTIONAL - if you hit OOM / want faster epochs, freeze the encoder:
# for p in model.model.encoder.parameters(): p.requires_grad_(False)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


params: 616M


## Data Augmentation: Boosting the Signal

Twenty-six thousand Bible verses provide a narrow linguistic register. To make the model more robust, we perform **Dictionary Augmentation**. By injecting 12,000 word-level pairs, we teach the model the building blocks of the language, helping it generalize beyond the formal structure of religious texts.

In [9]:
from datasets import Dataset

tok.src_lang = SRC_LANG
tok.tgt_lang = NEW_LANG

def build(src_texts, tgt_texts):
    return Dataset.from_dict({"english": list(src_texts), "pokot": list(tgt_texts)})

def encode(batch):
    inputs = [s for s in batch["english"]]
    targets = [t for t in batch["pokot"]]
    model_inputs = tok(inputs, max_length=MAX_LEN, truncation=True)

    with tok.as_target_tokenizer():
        labels = tok(targets, max_length=MAX_LEN, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def prep(ds):
    ds = ds.map(encode, batched=True, remove_columns=ds.column_names)
    return ds

train_ds = prep(build(train_df.english, train_df.pokot))
eval_sub = eval_df.sample(min(800, len(eval_df)), random_state=SEED).reset_index(drop=True)
eval_ds = prep(build(eval_sub.english, eval_sub.pokot))
print(f"train: {len(train_ds):,} | eval: {len(eval_ds):,}")

Map:   0%|          | 0/27848 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

train: 27,848 | eval: 800


In [10]:
extra = []
if USE_DICT_AUG:
    d = pd.read_csv(os.path.join(DATA_DIR, "pokot_dict.csv")).dropna(subset=["pokot", "english"])
    d["eng"] = (d.english.str.split(";").str[0].str.split(",").str[0].str.strip())
    d["pko"] = (d.pokot.str.split("/").str[0].str.split(",").str[0].str.strip())
    d = d[d.eng.str.len() > 1]
    d = d[~d.eng.str.contains(r"see|Eng\.|Kis\.|Turk\.|Kari\.|cf\.", regex=True, na=False)]
    extra += list(zip(d.pko, d.eng))
    print(f"dict pairs: {len(d):,}")

if USE_GRAMMAR_AUG:
    grammar_path = os.path.join(DATA_DIR, "pokot_grammar_pairs.csv")
    if os.path.exists(grammar_path):
        g = pd.read_csv(grammar_path).dropna(subset=["pokot", "english"])
        g = g[g.dict_hit >= 0.3]
        extra += list(zip(g.pokot, g.english)) * 3
        print(f"grammar pairs (x3): {len(g)*3}")
    else:
        print("Warning: pokot_grammar_pairs.csv not found, skipping grammar augmentation.")

if extra:
    from datasets import concatenate_datasets
    aug_ds = prep(build([e for p, e in extra], [p for p, e in extra]))
    train_ds = concatenate_datasets([train_ds, aug_ds]).shuffle(seed=SEED)
    print(f"train after augmentation: {len(train_ds):,} (+{len(aug_ds):,})")

dict pairs: 11,005


Map:   0%|          | 0/11005 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


train after augmentation: 38,853 (+11,005)


## The Training Phase: Precision Fine-Tuning

We are now ready for the heavy lifting. We use **chrF** (character n-gram F-score) as our primary metric, as it is far more reliable than BLEU for morphologically rich languages like Pokot.

To fit a 600M parameter model on a free T4 GPU, we utilize **Mixed Precision (FP16)** and **Gradient Checkpointing**. This setup allows us to maximize our batch size without crashing the runtime.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import sacrebleu
import torch

# Safeguard: Clear CUDA cache before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def postprocess(texts):
    return [t.strip() for t in texts]

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, PAD_ID)
    labels = np.where(labels != -100, labels, PAD_ID)
    pred_text = postprocess(tok.batch_decode(preds, skip_special_tokens=True))
    ref_text = postprocess(tok.batch_decode(labels, skip_special_tokens=True))
    return {"chrf": sacrebleu.corpus_chrf(pred_text, [ref_text]).score}

data_collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(OUT_DIR, "runs"),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    learning_rate=LR,
    num_train_epochs=1 if SMOKE_TEST else EPOCHS,
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=ACCUM,
    warmup_steps=WARMUP,
    fp16=True,
    gradient_checkpointing=True,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    report_to="none",
    seed=SEED,
    # Safeguard: optimize memory during evaluation
    per_device_eval_batch_size=BS,
    dataloader_num_workers=0,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tok, # Fixed: Using processing_class instead of deprecated tokenizer
)

try:
    trainer.train()
    trainer.save_model(os.path.join(OUT_DIR, "best"))
    tok.save_pretrained(os.path.join(OUT_DIR, "best"))
except RuntimeError as e:
    if "out of memory" in str(e):
        print("CUDA Out of Memory Error detected. Try reducing BS or MAX_LEN in the config cell.")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        raise e

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss


## The Moment of Truth: Live Inference

With training complete, let's put the model to the test. We'll feed it classic phrases and everyday sentences to see how well it translates English concepts into Pokot. This is where we see the fruits of our vocabulary expansion and data augmentation.

In [ ]:
model = trainer.model          # best checkpoint (load_best_model_at_end)
model.eval()

def translate(text, beams=4, max_new=256):
    ids = [SRC_ID] + tok(text, add_special_tokens=False).input_ids + [EOS_ID]
    x = torch.tensor([ids], device=model.device)
    with torch.no_grad():
        gen = model.generate(input_ids=x, forced_bos_token_id=TGT_ID,
                             num_beams=beams, max_new_tokens=max_new)
    return postprocess(tok.batch_decode(gen, skip_special_tokens=True))[0]

for s in ["In the beginning God created the heavens and the earth.",
          "The Lord is my shepherd; I have everything I need.",
          "Love is patient and kind.",
          "Go and make disciples of all nations.",
          "What time is it?",
          "I am going to the market tomorrow."]:
    print("EN:", s)
    print("PK:", translate(s))
    print()

## Rigorous Evaluation: Benchmarking Performance

True science requires objective measurement. We now run a full evaluation over the ~3,400 verses we held back at the start. Using a beam search of 4, we calculate the final chrF score to determine how close our model's outputs are to human-translated Pokot.

In [ ]:
FULL_EVAL = (not SMOKE_TEST)

model.generation_config.num_beams = 4
full_eval = prep(build(eval_df.english, eval_df.pokot))
if not FULL_EVAL:
    full_eval = full_eval.select(range(min(1000, len(full_eval))))
res = trainer.evaluate(eval_dataset=full_eval, metric_key_prefix="final")
print({k: round(v, 2) for k, v in res.items() if "chrf" in k})

In [ ]:
# side-by-side spot check
for _, r in eval_df.sample(8, random_state=7).iterrows():
    print(r.ref)
    print("  ref:", r.pokot[:110])
    print("  mt :", translate(r.english, beams=1)[:110])
    print()

## Deployment: Securing the Artifacts

Finally, we package our best-performing model weights and the custom tokenizer. By zipping these into Google Drive, the model is ready to be exported for downstream applications, such as a mobile translation app or a web-based dictionary.

In [ ]:
final_dir = os.path.join(OUT_DIR, "final")
trainer.model.save_pretrained(final_dir)
tok.save_pretrained(final_dir)
if IN_COLAB:
    zip_path = shutil.make_archive(os.path.join(OUT_DIR, "pokot_nllb_final"), "zip", final_dir)
    print("zipped:", zip_path)
print("saved:", final_dir)

## Troubleshooting

- **CUDA OOM (training)**: set `BS=4, ACCUM=8`; then `MAX_LEN=128`; then enable the encoder-freeze snippet in section 5. Grad checkpointing is already on.
- **OOM during eval**: `per_device_eval_batch_size=8` in section 7 args.
- **Colab session died mid-run**: checkpoints are in Drive at `pokot_nllb/runs/checkpoint-*`. Rerun cells 1-6 then `trainer.train(resume_from_checkpoint=True)`.
- **`eval_strategy` unknown**: your transformers is older than pinned - rerun the install cell, it pins 4.46.3.
- **chrF not improving after epoch ~4**: classic overfit on tiny corpora. Stop early, use the best checkpoint (Trainer already reloads it), or rerun with `EPOCHS=4` and `LR=1e-5`.

## Next steps (see ARCHITECTURE.md)

- Backtranslation round: train the reverse direction (same notebook, swap SRC/TGT), generate synthetic EN->PK pairs from Pokot mono text, mix in, retrain.
- Export to ONNX + int8 for CPU serving (Stage 3).
- Whisper: fine-tune `whisper-small` with LoRA on recorded Pokot audio for the voice pipeline (Stage P4).
- Realistic expectation: strong on Bible-register text, weaker on modern everyday input - the dict/grammar augmentation helps but a modern Pokot text collection is the single best upgrade.